# Bulk Collision Detection Analysis

This notebook processes Morpheus CPM simulation outputs to extract collision
statistics and cell tracking data. It scans a directory of simulation folders,
parses metadata from `model.xml`, extracts cell trajectories from
`celltracks.xml` (or falls back to image-based detection), and computes
collision metrics between the cell and track boundaries.

**Outputs:**
- `collision_analysis_summary.csv` — One row per simulation with aggregate metrics
- `collision_analysis_detailed.csv` — One row per frame with collision data
- Corresponding `.pkl` files for fast reloading

## Configuration

Edit the parameters below to match your data layout and analysis preferences.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Add the repository root to the path so we can import utils
REPO_ROOT = Path(os.getcwd()).parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    load_color_image,
    detect_black_pixels,
    detect_green_pixels,
    get_collision_intensity,
    get_cell_centroid,
    parse_model_xml,
    parse_celltracks_centroids,
    find_simulation_pngs,
    extract_model_type,
    extract_simulation_id,
    get_progress_from_centroids,
)
from utils.simulation_parser import extract_track_type_from_tiffs

print("Imports successful.")

In [ ]:
# ============================================================================
# USER CONFIGURATION — edit these to match your setup
# ============================================================================

# Path to the folder containing simulation subfolders
DATA_DIR = str(REPO_ROOT / "data" / "sample_simulations")

# Output directory for results
OUTPUT_DIR = str(REPO_ROOT / "output")

# Number of initial MCS timesteps to skip (cell settling period)
SKIP_INITIAL_MCS = 150

# Simulation IDs to exclude entirely (known bad runs)
EXCLUDE_SIMULATION_IDS = ['1205', '31', '1941']

# Expected model names (ordered by specificity)
EXPECTED_MODELS = ['WPI-PIP3', 'WPI', 'WP', 'Rac-Rho-T', 'Rac-Rho']

# Set True to append to existing results rather than re-processing
SKIP_ALREADY_ANALYZED = False

# Force re-analysis of specific IDs even when SKIP_ALREADY_ANALYZED is True
FORCE_REANALYZE_IDS = []

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Skip initial MCS: {SKIP_INITIAL_MCS}")


## Processing Functions

The core processing logic. `process_simulation_folder` handles a single
simulation folder, and `bulk_collision_analysis` orchestrates processing
across all folders.

In [ ]:
def process_simulation_folder(folder_path):
    """Process one simulation folder and return summary + detailed data.

    Extracts metadata from model.xml, cell trajectories from celltracks.xml
    (or image-based fallback), and computes collision metrics for each frame.

    Returns:
        Tuple of (summary_dict, detailed_rows_list), or (None, []) on failure.
    """
    folder_name = os.path.basename(folder_path)
    sim_id = extract_simulation_id(folder_name)
    model_type = extract_model_type(folder_name, EXPECTED_MODELS)

    if sim_id in EXCLUDE_SIMULATION_IDS:
        return None, []

    # --- Parse model.xml for metadata ---
    model_path = os.path.join(folder_path, 'model.xml')
    meta = parse_model_xml(model_path)

    # Fall back to TIFF filenames for track type if not in model.xml
    if not meta['track_type']:
        meta['track_type'] = extract_track_type_from_tiffs(folder_path)

    # --- Discover PNG frames ---
    try:
        png_files = find_simulation_pngs(folder_path)
        if len(png_files) == 0:
            print(f"  No valid PNGs in {folder_name}, skipping.")
            return None, []
    except Exception as e:
        print(f"  Error finding PNGs in {folder_name}: {e}")
        return None, []

    # --- Cell centroid tracking ---
    celltracks_path = os.path.join(folder_path, 'celltracks.xml')
    if os.path.exists(celltracks_path):
        centroids = parse_celltracks_centroids(celltracks_path)
        progress_data_source = 'celltracks'
        height_for_progress = meta['height']
    else:
        # Fallback: compute centroids from images
        centroids = []
        for t, fp in png_files:
            img = load_color_image(fp)
            cx, cy = get_cell_centroid(img)
            centroids.append((t, cx, cy))
        progress_data_source = 'images'
        if png_files:
            first_img = load_color_image(png_files[0][1])
            height_for_progress = str(first_img.shape[0])
        else:
            height_for_progress = meta['height']

    # --- Track progress ---
    # Filter centroids to only include up to the max timestep available in PNGs
    if png_files:
        max_png_timestep = max(t for t, fp in png_files)
        centroids = [c for c in centroids if c[0] <= max_png_timestep]
        
    max_y_position, track_progress = get_progress_from_centroids(
        centroids, height_for_progress
    )

    # --- Build centroid lookup for detailed data ---
    centroid_lookup = {t: (x, y) for t, x, y in centroids}

    # --- Collision analysis ---
    total_frames = len(png_files)
    frames_skipped = sum(1 for t, fp in png_files if t < SKIP_INITIAL_MCS)
    frames_analyzed = max(0, total_frames - frames_skipped)
    collision_frames = 0
    collision_intensities = []
    detailed_rows = []

    for i, (timestep, filepath) in enumerate(png_files):
        image = load_color_image(filepath)
        collision_occurred, intensity, _ = get_collision_intensity(image)

        # Build detailed row
        row = {
            'simulation_id': sim_id,
            'folder_name': folder_name,
            'model_type': model_type,
            'timestep': timestep,
            'frame_index': i,
            'collision': collision_occurred,
            'collision_intensity': intensity,
        }

        # Add centroid data
        if timestep in centroid_lookup:
            row['centroid_x'] = centroid_lookup[timestep][0]
            row['centroid_y'] = centroid_lookup[timestep][1]
        elif progress_data_source == 'images':
            cx, cy = get_cell_centroid(image)
            row['centroid_x'] = cx
            row['centroid_y'] = cy

        detailed_rows.append(row)

        # Accumulate collision stats (only for frames after skip period)
        if timestep >= SKIP_INITIAL_MCS:
            if collision_occurred:
                collision_frames += 1
            collision_intensities.append(intensity)

    collision_pct = (
        (collision_frames / frames_analyzed) * 100
        if frames_analyzed > 0 else 0.0
    )

    summary = {
        'simulation_id': sim_id,
        'folder_name': folder_name,
        'model_type': model_type,
        'track_type': meta['track_type'],
        'domain_width': meta['width'],
        'domain_height': meta['height'],
        'domain_file': meta['domain_file'],
        'gradient_file': meta['gradient_file'],
        'gradient_scale': meta['gradient_scale'],
        'mcs_duration': meta['mcs_duration'],
        'folder_path': folder_path,
        'png_count': len(png_files),
        'total_frames': total_frames,
        'frames_analyzed': frames_analyzed,
        'frames_skipped': frames_skipped,
        'collision_frames': collision_frames,
        'collision_percentage': collision_pct,
        'max_collision_intensity': max(collision_intensities) if collision_intensities else 0.0,
        'mean_collision_intensity': np.mean(collision_intensities) if collision_intensities else 0.0,
        'track_progress_percentage': track_progress,
        'max_y_position': max_y_position,
        'progress_data_source': progress_data_source,
    }

    return summary, detailed_rows


In [ ]:
def bulk_collision_analysis(data_dir):
    """Process all simulation folders in the given directory.

    Supports incremental processing: if SKIP_ALREADY_ANALYZED is True,
    previously analyzed simulations are skipped unless their IDs appear
    in FORCE_REANALYZE_IDS.

    Returns:
        Tuple of (summary_df, detailed_df).
    """
    all_folders = sorted([
        os.path.join(data_dir, d)
        for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])

    # Check for previously analyzed simulations
    already_done_ids = set()
    summary_path = Path(OUTPUT_DIR) / 'collision_analysis_summary.csv'
    if SKIP_ALREADY_ANALYZED and summary_path.exists():
        try:
            prev_df = pd.read_csv(summary_path, usecols=['simulation_id'])
            already_done_ids = set(prev_df['simulation_id'].astype(str).unique())
            print(f"Found {len(already_done_ids)} previously analyzed simulations.")
        except Exception as e:
            print(f"Warning: Could not read previous summary: {e}")

    # Determine which folders to process
    folders_to_process = []
    for folder in all_folders:
        sim_id = extract_simulation_id(os.path.basename(folder))
        if sim_id in EXCLUDE_SIMULATION_IDS:
            continue
        if (SKIP_ALREADY_ANALYZED
                and sim_id in already_done_ids
                and sim_id not in FORCE_REANALYZE_IDS):
            continue
        folders_to_process.append(folder)

    total = len(folders_to_process)
    print(f"Processing {total} simulation(s)...")

    summary_rows = []
    detailed_rows = []

    for idx, folder in enumerate(folders_to_process, 1):
        folder_name = os.path.basename(folder)
        print(f"\n[{idx}/{total}] {folder_name}")

        summary, details = process_simulation_folder(folder)
        if summary is not None:
            summary_rows.append(summary)
            detailed_rows.extend(details)

    summary_df = pd.DataFrame(summary_rows)
    detailed_df = pd.DataFrame(detailed_rows)
    return summary_df, detailed_df

## Run the Analysis

In [ ]:
summary_df, detailed_df = bulk_collision_analysis(DATA_DIR)
print(f"\nProcessed {len(summary_df)} simulations, {len(detailed_df)} frames.")

## Save Results

Results are saved in both CSV (human-readable) and pickle (fast reload) formats.
If `SKIP_ALREADY_ANALYZED` is True, new results are appended to existing files.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

summary_path = Path(OUTPUT_DIR) / 'collision_analysis_summary.csv'
detailed_path = Path(OUTPUT_DIR) / 'collision_analysis_detailed.csv'

# Append to existing results if incremental mode is on
if SKIP_ALREADY_ANALYZED and summary_path.exists():
    try:
        prev_summary = pd.read_csv(summary_path)
        if FORCE_REANALYZE_IDS:
            prev_summary = prev_summary[
                ~prev_summary['simulation_id'].astype(str).isin(FORCE_REANALYZE_IDS)
            ]
        summary_df = pd.concat([prev_summary, summary_df], ignore_index=True)
        summary_df = summary_df.drop_duplicates(
            subset=['simulation_id'], keep='last'
        ).reset_index(drop=True)
    except Exception as e:
        print(f"Warning: could not append to summary: {e}")

if SKIP_ALREADY_ANALYZED and detailed_path.exists():
    try:
        prev_detailed = pd.read_csv(detailed_path)
        if FORCE_REANALYZE_IDS:
            prev_detailed = prev_detailed[
                ~prev_detailed['simulation_id'].astype(str).isin(FORCE_REANALYZE_IDS)
            ]
        detailed_df = pd.concat([prev_detailed, detailed_df], ignore_index=True)
        if 'timestep' in detailed_df.columns:
            detailed_df = detailed_df.drop_duplicates(
                subset=['simulation_id', 'timestep'], keep='last'
            ).reset_index(drop=True)
    except Exception as e:
        print(f"Warning: could not append to detailed: {e}")

# Save CSV
summary_df.to_csv(summary_path, index=False)
detailed_df.to_csv(detailed_path, index=False)

# Save pickle for fast reloading
summary_df.to_pickle(summary_path.with_suffix('.pkl'))
detailed_df.to_pickle(detailed_path.with_suffix('.pkl'))

print(f"Summary saved: {summary_path} ({len(summary_df)} rows)")
print(f"Detailed saved: {detailed_path} ({len(detailed_df)} rows)")

## Quick Summary Report

In [ ]:
if len(summary_df) > 0:
    print("=" * 60)
    print("COLLISION DETECTION ANALYSIS SUMMARY")
    print("=" * 60)
    print(f"Total simulations: {len(summary_df)}")
    print(f"Total frames: {len(detailed_df)}")
    print(f"Model types: {', '.join(summary_df['model_type'].unique())}")
    print(f"Track types: {', '.join(summary_df['track_type'].dropna().unique())}")
    print()

    stats = summary_df.groupby('model_type').agg({
        'collision_percentage': ['mean', 'std', 'count'],
        'track_progress_percentage': ['mean'],
    }).round(2)

    print("Collision Statistics by Model Type:")
    print(stats)
else:
    print("No simulations were processed.")